# Fig 7: TTFT Slowdown from Power-Capping (300W vs 200W)

**Requires sudo:** `sudo nvidia-smi -i <GPU_ID> -pl 200` to set the 200W power cap before running the collect cell.  
**Output:** `paper/figures/section3/output/prefill_powercap_ratio.{pdf,png}`

### Call order
1. Collect 300W cache-cost data (see Fig 2 notebook): `profiling/run_cache_cost.py` → `scripts/plot_cache_cost.py`
2. Collect 200W cache-cost data (same harness, GPU at 200W): `profiling/run_cache_cost.py --out output/200W/cache_cost_data` → `scripts/plot_cache_cost.py --dir output/200W`
3. `scripts/plot_prefill_powercap_ratio.py`

Steps 1–2 collect raw data and only need to be run once. If data is available from another machine, copy `cache_cost_data/` into both `paper/figures/section3/output/300W/` and `paper/figures/section3/output/200W/` and skip the collect cell below.

In [ ]:
import subprocess
from pathlib import Path

REPO_ROOT = next(
    p for p in [Path.cwd()] + list(Path.cwd().parents)
    if (p / ".conserve_root").exists()
)


def run(cmd):
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(result.stdout, end="")
    if result.returncode != 0:
        raise RuntimeError(f"Command failed (exit {result.returncode}): {' '.join(str(c) for c in cmd)}")

In [ ]:
# Steps 1-2: collect raw cache-cost data at both power levels. Skip if data already exists.
# Note: 300W data is also required by Fig 2 — see that notebook to collect it.
# For 200W: set GPU power cap before running (requires sudo):
#   sudo nvidia-smi -i <GPU_ID> -pl 200
out_200w = REPO_ROOT / "paper/figures/section3/output/200W/cache_cost_data"
run(["python", str(REPO_ROOT / "profiling/run_cache_cost.py"), "--out", str(out_200w)])
run([
    "python", str(REPO_ROOT / "paper/figures/section3/scripts/plot_cache_cost.py"),
    "--dir", str(REPO_ROOT / "paper/figures/section3/output/200W"),
])

In [ ]:
# Step 3: plot
%matplotlib inline
%run ../scripts/plot_prefill_powercap_ratio.py

In [ ]:
from IPython.display import Image

Image(str(REPO_ROOT / "paper/figures/section3/output/prefill_powercap_ratio.png"))